In [1]:
import pandas as pd
import numpy as np
import re
import torch
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, SimpleRNN, Dense
from tensorflow.keras.optimizers import Adam
from sklearn.model_selection import train_test_split

In [2]:
data = pd.read_csv("data/dataset_annotated_balanced.csv")

In [3]:
data.head()

,Joke,Joke_Cleaned,Joke_Normalized,Joke_Length_Chars,Joke_Length_Words,Offensive_Label
0,What did the hillbilly say to his sister after...,what did the hillbilly say to his sister after...,what did the hillbilly say to his sister after...,97,20,1
1,What's the difference Donald Trump and my Vagi...,what's the difference donald trump and my vagi...,what's the difference donald trump and my vagi...,91,17,1
2,What did the man with The World's Largest Peni...,what did the man with the world's largest peni...,what did the man with the world's largest peni...,130,26,1
3,I lost my virginity to a retarded girl last ni...,i lost my virginity to a retarded girl last ni...,i lost my virginity to a retarded girl last ni...,76,16,1
4,What did the gay guy say to his lover when the...,what did the gay guy say to his lover when the...,what did the gay guy say to his lover when the...,106,23,1


In [ ]:
# Ne garder que les blagues offensives
data_offensive = data[data["Offensive_Label"] == 1].copy()

# Liste de mots offensants
offensive_words = set([
    "sex", "gender", "womanizer", "feminist", "chauvinist", "misogynist",
    "misandrist", "patriarchy", "matriarchy",
    "racism", "racist", "black", "white", "asian", "african", "indian",
    "nigger", "cracker", "slur", "latino", "hispanic", "jew", "antisemitic",
    "arab", "middle eastern", "xenophobia", "immigrant",
    "penis", "vagina", "breast", "boobs", "dick", "cock", "pussy", "butt",
    "anus", "testicle", "scrotum", "nipple", "semen", "sperm", "condom",
    "masturbate", "ejaculate", "rape", "molest", "abuse", "hooker", 
    "prostitute", "whore", "slut", "porn", "pornography", "fetish",
    "orgy", "orgasm", "clitoris", "labia", "erection", "virginity",
    "dumb", "idiot", "moron", "retard", "cripple",
    "gay", "lesbian", "homosexual", "bisexual", "transgender", 
    "queer", "homo", "tranny", "dyke", "fag", "pervert", 
    "harass", "molestation", "pedophile", "incest"
])

def calculate_offensive_score(joke):
    words = re.findall(r'\b\w+\b', str(joke).lower())  # Tokenisation simple
    total_words = len(words)
    if total_words == 0:
        return 0  # Éviter la division par zéro

    nb_offensive = sum(1 for word in words if word in offensive_words)
    offensive_ratio = nb_offensive / total_words  # Ratio offensif

    # Échelle plus progressive
    if offensive_ratio == 0:
        return 0
    elif offensive_ratio < 0.1:
        return 1
    elif offensive_ratio < 0.2:
        return 2
    elif offensive_ratio < 0.3:
        return 3
    elif offensive_ratio < 0.4:
        return 4
    else:
        return 5



In [5]:
# Appliquer la fonction au dataset
data_offensive["Offensive_Score"] = data_offensive["Joke_Normalized"].apply(calculate_offensive_score)

In [20]:
data_offensive.head()

,Joke,Joke_Cleaned,Joke_Normalized,Joke_Length_Chars,Joke_Length_Words,Offensive_Label,Offensive_Score
0,What did the hillbilly say to his sister after...,what did the hillbilly say to his sister after...,what did the hillbilly say to his sister after...,97,20,1,2
1,What's the difference Donald Trump and my Vagi...,what's the difference donald trump and my vagi...,what's the difference donald trump and my vagi...,91,17,1,1
2,What did the man with The World's Largest Peni...,what did the man with the world's largest peni...,what did the man with the world's largest peni...,130,26,1,1
3,I lost my virginity to a retarded girl last ni...,i lost my virginity to a retarded girl last ni...,i lost my virginity to a retarded girl last ni...,76,16,1,1
4,What did the gay guy say to his lover when the...,what did the gay guy say to his lover when the...,what did the gay guy say to his lover when the...,106,23,1,1


In [19]:
data_offensive['Offensive_Score'].unique()
data_offensive["Offensive_Score"].value_counts()


Offensive_Score
1    5046
2    1971
3     217
4      39
0      21
5      11
Name: count, dtype: int64

RNN pour la régression du score d'offensivité

In [6]:
print(torch.cuda.is_available())
print(torch.cuda.device_count())
print(torch.cuda.get_device_name(0))

True
1
NVIDIA GeForce RTX 3070 Laptop GPU


In [7]:
# Paramètres du modèle
MAX_VOCAB_SIZE = 10000  # Nombre max de mots à prendre en compte
MAX_SEQUENCE_LENGTH = 100  # Longueur max des séquences
EMBEDDING_DIM = 128  # Taille des embeddings

# Tokenization et padding des blagues
tokenizer = Tokenizer(num_words=MAX_VOCAB_SIZE, oov_token="<OOV>")
tokenizer.fit_on_texts(data_offensive["Joke_Normalized"])
sequences = tokenizer.texts_to_sequences(data_offensive["Joke_Normalized"])
padded_sequences = pad_sequences(sequences, maxlen=MAX_SEQUENCE_LENGTH, padding="post")

# Séparation train/test
X_train, X_test, y_train, y_test = train_test_split(padded_sequences, data_offensive["Offensive_Score"], test_size=0.2, random_state=42)

# Conversion en numpy array (float32 pour éviter les erreurs TensorFlow)
X_train, X_test = np.array(X_train), np.array(X_test)
y_train, y_test = np.array(y_train, dtype=np.float32), np.array(y_test, dtype=np.float32)

In [8]:
# Création du modèle RNN
model = Sequential([
    Embedding(input_dim=MAX_VOCAB_SIZE, output_dim=EMBEDDING_DIM, input_length=MAX_SEQUENCE_LENGTH),
    SimpleRNN(128, return_sequences=True),  # Couche RNN
    SimpleRNN(64, return_sequences=False),  # Une deuxième couche RNN
    Dense(64, activation="relu"),
    Dense(32, activation="relu"),
    Dense(1, activation="linear")  
])

# Compilation du modèle
model.compile(loss="mse", optimizer=Adam(learning_rate=0.001), metrics=["mae"])


c:\Users\Pc\AppData\Local\Programs\Python\Python312\Lib\site-packages\keras\src\layers\core\embedding.py:90: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


In [9]:
# Entraînement du modèle
history = model.fit(X_train, y_train, epochs=10, batch_size=32, validation_data=(X_test, y_test))

Epoch 1/10
183/183 ━━━━━━━━━━━━━━━━━━━━ 15s 53ms/step - loss: 0.4968 - mae: 0.5439 - val_loss: 0.3407 - val_mae: 0.5298
Epoch 2/10
183/183 ━━━━━━━━━━━━━━━━━━━━ 9s 48ms/step - loss: 0.3425 - mae: 0.4862 - val_loss: 0.3515 - val_mae: 0.4405
Epoch 3/10
183/183 ━━━━━━━━━━━━━━━━━━━━ 9s 48ms/step - loss: 0.3574 - mae: 0.4812 - val_loss: 0.3355 - val_mae: 0.4676
Epoch 4/10
183/183 ━━━━━━━━━━━━━━━━━━━━ 9s 48ms/step - loss: 0.3515 - mae: 0.4818 - val_loss: 0.3337 - val_mae: 0.4727
Epoch 5/10
183/183 ━━━━━━━━━━━━━━━━━━━━ 9s 49ms/step - loss: 0.3614 - mae: 0.4920 - val_loss: 0.3430 - val_mae: 0.4526
Epoch 6/10
183/183 ━━━━━━━━━━━━━━━━━━━━ 10s 54ms/step - loss: 0.3386 - mae: 0.4748 - val_loss: 0.3308 - val_mae: 0.4989
Epoch 7/10
183/183 ━━━━━━━━━━━━━━━━━━━━ 10s 55ms/step - loss: 0.3459 - mae: 0.4960 - val_loss: 0.3347 - val_mae: 0.4696
Epoch 8/10
183/183 ━━━━━━━━━━━━━━━━━━━━ 10s 55ms/step - loss: 0.3286 - mae: 0.4793 - val_loss: 0.3321 - val_mae: 0.5074
Epoch 9/10
183/183 ━━━━━━━━━━━━━━━━━━━━ 10s 